# 17 — Final Model Training and Artifact Preparation

This notebook finalizes the selected fraud detection model workflow and prepares the project for inference and FastAPI integration. The focus is production readiness: clear paths, final artifact locations, and a clean setup for training and saving the validated model in later phases.


## Purpose

This notebook exists to prepare the final training and artifact-saving workflow for the fraud detection system.

It will eventually be responsible for:

- Training the final selected fraud detection model
- Reusing the final decision policy from previous notebooks
- Saving model artifacts needed for inference
- Saving feature columns, metrics, metadata, and decision policy
- Preparing the project for the next inference/API stage

This notebook should not repeat full EDA, full model comparison, or threshold tuning. Those tasks were completed earlier in the project. Notebook 17 is focused on turning the validated modeling decisions into reusable production-style artifacts.


## Previous Notebook Context

The project workflow leading into this notebook is:

- `13_model_training.ipynb` trained candidate models.
- `14_model_evaluation.ipynb` evaluated model performance and confirmed the strongest model choice.
- `15_threshold_tuning.ipynb` selected suitable fraud probability thresholds using validation/OOF predictions and holdout confirmation.
- `16_decision_logic.ipynb` converted model fraud probabilities into `APPROVE`, `REVIEW`, and `BLOCK` decisions.
- `17_final_model_training.ipynb` now prepares the final validated model training and artifact-saving workflow.

The difference between notebook 13 and notebook 17 is important. Notebook 13 is for experimentation and candidate model training. Notebook 17 is for final validated model training and artifact preparation for inference/API use. In other words, notebook 13 helps decide what works; notebook 17 prepares the final version for reuse by downstream code.


## Imports

This section imports the libraries needed for Phase 1 and the later final-training phases. The imports are intentionally placed up front so later phases can train the final model, compute metrics, and save artifacts without changing the notebook setup.


In [1]:
import json
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


## Path Configuration

This section defines all important input and output paths with `pathlib.Path`. The project root logic works whether the notebook is run from the repository root or from inside the `notebooks/` folder.

Phase 1 only configures paths. It does not load the dataset, train a model, evaluate a model, or save final model artifacts.


In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
REPORTS_DIR = PROJECT_ROOT / "reports"
TABLES_DIR = REPORTS_DIR / "tables" / "17_final_model_training"
FIGURES_DIR = REPORTS_DIR / "figures" / "17_final_model_training"

# Existing processed dataset selected for final model training setup.
# The repository currently contains final_features.csv rather than creditcard_model_ready.csv.
FINAL_DATASET_PATH = DATA_PROCESSED_DIR / "final_features.csv"
DECISION_POLICY_PATH = ARTIFACTS_DIR / "decision_policy.json"

FINAL_MODEL_PATH = ARTIFACTS_DIR / "final_validated_fraud_model.joblib"
FINAL_FEATURE_COLUMNS_PATH = ARTIFACTS_DIR / "final_feature_columns.json"
FINAL_MODEL_METADATA_PATH = ARTIFACTS_DIR / "final_model_metadata.json"
FINAL_MODEL_METRICS_PATH = ARTIFACTS_DIR / "final_model_metrics.json"
FINAL_DECISION_POLICY_PATH = ARTIFACTS_DIR / "final_decision_policy.json"


## Output Folder Creation

This section creates the folders that later phases will use for artifacts, tables, and figures. Creating them now makes the notebook environment easy to verify before any final model training begins.


In [3]:
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

configured_paths = {
    "Project root": PROJECT_ROOT,
    "Processed data directory": DATA_PROCESSED_DIR,
    "Artifacts directory": ARTIFACTS_DIR,
    "Tables output directory": TABLES_DIR,
    "Figures output directory": FIGURES_DIR,
    "Final dataset path": FINAL_DATASET_PATH,
    "Decision policy path": DECISION_POLICY_PATH,
    "Final model artifact path": FINAL_MODEL_PATH,
}

print("Configured Phase 1 paths:")
for label, path in configured_paths.items():
    print(f"- {label}: {path}")


Configured Phase 1 paths:
- Project root: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection
- Processed data directory: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/data/processed
- Artifacts directory: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts
- Tables output directory: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/reports/tables/17_final_model_training
- Figures output directory: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/reports/figures/17_final_model_training
- Final dataset path: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/data/processed/final_features.csv
- Decision policy path: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/decision_policy.json
- Final model artifact path: /Users/mohammadmubashir/VCode/Credit-Card-Fraud-Detection/artifacts/final_validated_fraud_model.joblib


## Phase 1 Stop and Check

Before moving to Phase 2, confirm the setup questions below:

- Do I know the input dataset path?
- Do I know where final model artifacts will be saved?
- Do I know where reports and tables will be saved?
- Do I understand the difference between notebook 13 and notebook 17?
- Did the required output folders get created successfully?

Phase 1 stops here intentionally. The notebook environment is configured, but no data has been loaded, no model has been trained, no evaluation has been run, and no final model artifacts have been saved yet.
